<a href="https://colab.research.google.com/github/lucaslucena-lab/impacta-labs/blob/main/youtube_telegram_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📹 Sistema de Resumo de Vídeos YouTube → Telegram

## 👥 Integrantes do Grupo

- **Integrante 1:** [Nome Completo]
- **Integrante 2:** [Nome Completo]
- **Integrante 3:** [Nome Completo]
- **Integrante 4:** [Nome Completo]

---

## 📋 Descrição do Projeto

Este notebook automatiza o processo de:
1. Receber link de canal do YouTube via Telegram
2. Capturar os 5 últimos vídeos do canal
3. Extrair transcrições dos vídeos
4. Gerar resumos com Gemini AI
5. Criar áudio com os resumos
6. Enviar áudio de volta via Telegram

## 🔧 Instalação de Dependências

In [1]:
!pip install -q google-generativeai google-genai python-telegram-bot youtube-transcript-api feedparser requests

## 🔑 Configuração de Credenciais

**IMPORTANTE:** Configure suas chaves de API aqui

In [2]:
import os
from google.colab import userdata

# Secrets do Colab

try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
    YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Credenciais carregadas dos secrets do Colab")
except:
    print("⚠️ Secrets não encontrados. Configure manualmente:")

# Variável dinâmica do canal
CANAL_YOUTUBE = "https://www.youtube.com/@PrimoCast"

✅ Credenciais carregadas dos secrets do Colab


## 📚 Imports e Configurações

In [3]:
import re
import requests
import feedparser
import google.generativeai as genai
from youtube_transcript_api import YouTubeTranscriptApi
from telegram import Bot, Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
import asyncio
from IPython.display import Audio, display
import nest_asyncio
from google.genai.types import Content, Part
import getpass

# Permitir loops assíncronos no Colab
nest_asyncio.apply()

# Configurar Gemini
genai.configure(api_key=GEMINI_API_KEY)

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


## 🔍 Função 1: Extrair Channel ID do YouTube

In [4]:
def extrair_channel_id(url_canal):
    """
    Extrai o Channel ID de uma URL do YouTube
    Suporta formatos: @username, /channel/ID, /c/custom
    """
    print(f"🔍 Processando URL: {url_canal}")

    # Se já for um channel ID direto
    if '/channel/' in url_canal:
        channel_id = url_canal.split('/channel/')[-1].split('/')[0]
        print(f"✅ Channel ID encontrado: {channel_id}")
        return channel_id

    # Se for @username ou /c/custom, usar API do YouTube
    username_match = re.search(r'@([\w-]+)', url_canal)
    custom_match = re.search(r'/c/([\w-]+)', url_canal)

    if username_match:
        username = username_match.group(1)
        api_url = f"https://www.googleapis.com/youtube/v3/channels?part=id&forHandle={username}&key={YOUTUBE_API_KEY}"
    elif custom_match:
        custom_name = custom_match.group(1)
        api_url = f"https://www.googleapis.com/youtube/v3/channels?part=id&forUsername={custom_name}&key={YOUTUBE_API_KEY}"
    else:
        raise ValueError("❌ Formato de URL não reconhecido")

    response = requests.get(api_url)
    data = response.json()

    if 'items' in data and len(data['items']) > 0:
        channel_id = data['items'][0]['id']
        print(f"✅ Channel ID encontrado: {channel_id}")
        return channel_id
    else:
        raise ValueError(f"❌ Canal não encontrado. Resposta da API: {data}")

# Teste
channel_id = extrair_channel_id(CANAL_YOUTUBE)
print(f"\n📌 Channel ID: {channel_id}")

🔍 Processando URL: https://www.youtube.com/@PrimoCast
✅ Channel ID encontrado: UCfMA8s_QXPPcuOTjTDFb4vA

📌 Channel ID: UCfMA8s_QXPPcuOTjTDFb4vA


## 📡 Função 2: Capturar Feed RSS e Últimos 5 Vídeos

In [5]:
def obter_ultimos_videos(channel_id, quantidade=5):
    """
    Obtém os últimos vídeos de um canal via RSS
    """
    rss_url = f"https://www.youtube.com/feeds/videos.xml?channel_id={channel_id}"
    print(f"📡 Acessando RSS: {rss_url}")

    feed = feedparser.parse(rss_url)

    if not feed.entries:
        raise ValueError("❌ Nenhum vídeo encontrado no feed")

    videos = []
    for entry in feed.entries[:quantidade]:
        video_id = entry.yt_videoid
        titulo = entry.title
        videos.append({
            'video_id': video_id,
            'titulo': titulo,
            'url': f"https://www.youtube.com/watch?v={video_id}"
        })
        print(f"  📹 {titulo} ({video_id})")

    print(f"\n✅ {len(videos)} vídeos capturados")
    return videos

# Teste
videos = obter_ultimos_videos(channel_id)
for i, video in enumerate(videos, 1):
    print(f"{i}. {video['titulo']}")

📡 Acessando RSS: https://www.youtube.com/feeds/videos.xml?channel_id=UCfMA8s_QXPPcuOTjTDFb4vA
  📹 COMO GANHAR DINHEIRO NA INTERNET? - CLIQUE PARA ASSISTIR #shorts (LuoaflbuHtg)
  📹 COMO GANHAR DINHEIRO COM MARKETING DIGITAL? - CLIQUE PARA ASSISTIR #shorts (-do-2NsYxmI)
  📹 AINDA DÁ TEMPO DE COMEÇAR NO MARKETING DIGITAL? (IxOig0DzNxM)
  📹 o SEGREDO PRA MUDAR DE VIDA é NÃO TER MEDO DE ARRISCAR (AH5Ztc11QJc)
  📹 VENDA PRODUTOS EM ESPANHOL E FATURE MILHÕES - CLIQUE PARA ASSISTIR #shorts (2NTf-WPlcfI)

✅ 5 vídeos capturados
1. COMO GANHAR DINHEIRO NA INTERNET? - CLIQUE PARA ASSISTIR #shorts
2. COMO GANHAR DINHEIRO COM MARKETING DIGITAL? - CLIQUE PARA ASSISTIR #shorts
3. AINDA DÁ TEMPO DE COMEÇAR NO MARKETING DIGITAL?
4. o SEGREDO PRA MUDAR DE VIDA é NÃO TER MEDO DE ARRISCAR
5. VENDA PRODUTOS EM ESPANHOL E FATURE MILHÕES - CLIQUE PARA ASSISTIR #shorts


## 📝 Função 3: Extrair Transcrições

In [6]:
def obter_transcricao(video_id):
    """
    Obtém a transcrição de um vídeo do YouTube
    Tenta português primeiro, depois inglês, depois qualquer idioma
    """
    try:
        # Tentar português
        # 1. Criar instância
        api = YouTubeTranscriptApi()

        # 2. Obter lista de transcrições
        transcript_list = api.list(video_id)

        # 3. Encontrar transcrição no idioma desejado
        transcript = transcript_list.find_transcript(['pt', 'pt-BR'])

        # 4. Buscar o conteúdo
        content = transcript.fetch()

        # 5. Acessar .text como ATRIBUTO (não dicionário)
        texto = ' '.join([entry.text for entry in content])
        return texto
    except:
        try:
            # Tentar inglês
            transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
            texto = ' '.join([entry['text'] for entry in transcript])
            return texto
        except:
            try:
                # Tentar qualquer idioma disponível
                transcript = YouTubeTranscriptApi.get_transcript(video_id)
                texto = ' '.join([entry['text'] for entry in transcript])
                return texto
            except Exception as e:
                return f"[Transcrição não disponível: {str(e)}]"

# Teste com primeiro vídeo
print("📝 Testando transcrição do primeiro vídeo...\n")
transcricao_teste = obter_transcricao(videos[0]['video_id'])
print(f"Primeiros 500 caracteres:\n{transcricao_teste[:500]}...")

📝 Testando transcrição do primeiro vídeo...

Primeiros 500 caracteres:
Tanta gente conseguiu, por você não? Tanta gente normal, comum, não tô falando de grande gênos. Fala que o digital é tipo jogador de futebol. Tem um jogador de futebol da Série B que você vai olhar lá o cara ganha um salário de 70.000 digital é assim também, cara. Tem uma pessoa ali que ninguém sabe quem é. Olha ali, cara. O cara tá ganhando 40 45. >> Pô, vocês mostraram vários aí, >> cara. Mas até até mesmo nesses números, eu acho até que a galera nem deveria mirar nesse número. Eu acho que a g...


## 🤖 Função 4: Resumir com Gemini

In [11]:
def resumir_com_gemini(transcricao, titulo):
    """
    Usa Gemini para criar resumo conciso da transcrição
    """
    model = genai.GenerativeModel('gemini-2.5-flash')

    prompt = f"""
    Você é um assistente especializado em resumir conteúdo de vídeos.

    Título do vídeo: {titulo}

    Transcrição:
    {transcricao}

    Por favor, crie um resumo conciso e informativo deste vídeo em até 3 parágrafos,
    destacando os pontos principais e informações mais relevantes.
    """

    response = model.generate_content(prompt)
    return response.text

# Teste
print("🤖 Testando resumo com Gemini...\n")
resumo_teste = resumir_com_gemini(transcricao_teste, videos[0]['titulo'])
print(f"Resumo:\n{resumo_teste}")

🤖 Testando resumo com Gemini...

Resumo:
Este vídeo destaca que ganhar dinheiro na internet é uma possibilidade real e acessível para qualquer pessoa comum, não apenas para grandes gênios. Utiliza a analogia de jogadores de futebol da Série B que ganham altos salários para ilustrar que, no digital, existem indivíduos que, mesmo desconhecidos, faturam quantias expressivas, como R$40 mil ou R$45 mil.

Contudo, o palestrante aconselha a não focar inicialmente nesses números grandiosos, mas sim em objetivos mais palpáveis. A sugestão é tentar replicar na internet a renda de um emprego que não agrada – por exemplo, alcançar R$2.000 online em vez de R$2.000 em um trabalho convencional – como uma meta inicial e motivadora.

Para reforçar essa ideia de que grandes conquistas são possíveis, o narrador compartilha sua própria experiência. Ele revela que, vindo da periferia e sem conhecer ninguém com R$10 mil guardados, considerava impossível juntar R$100 mil, até que ele próprio conseguiu atingi

In [13]:
from google import genai
from google.genai.types import Content, Part
import getpass

client = genai.Client(api_key=GEMINI_API_KEY)

def resumir_com_gemini(transcricao: str, titulo: str) -> str:
    prompt = f"""Você é um assistente que resume vídeos.
Título: {titulo}

Transcrição:
{transcricao}

Resuma em até 3 parágrafos, focando nos pontos principais e informações mais relevantes."""

    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return resp.text

print("🤖 Testando resumo com Gemini (SDK nova)...\n")
print(resumir_com_gemini(transcricao_teste, videos[0]['titulo']))


🤖 Testando resumo com Gemini (SDK nova)...

O vídeo destaca que o sucesso na internet é acessível a muitas pessoas, mesmo aquelas consideradas "normais" ou "comuns", comparando o potencial de ganhos no digital ao salário de jogadores de futebol da Série B, que faturam alto. Segundo o palestrante, existem indivíduos no mercado digital que, apesar de pouco conhecidos, estão ganhando quantias significativas como R$40.000 ou R$45.000.

A principal orientação é focar em metas mais realistas e alcançáveis. Em vez de mirar imediatamente em números muito altos, a sugestão é que se alguém ganha, por exemplo, R$2.000 em um emprego que não aprecia, o objetivo inicial deveria ser replicar essa mesma renda de R$2.000 através das oportunidades online, tornando a transição mais palpável.

O palestrante compartilha sua experiência pessoal, revelando que antes considerava "impossível" juntar R$100.000, especialmente vindo da periferia onde não conhecia ninguém com R$10.000 guardados. No entanto, ele co

## 🎯 Função 5: Processar Todos os Vídeos

In [14]:
def processar_videos(videos):
    """
    Processa todos os vídeos: transcrição + resumo
    """
    resumos = []

    for i, video in enumerate(videos, 1):
        print(f"\n{'='*60}")
        print(f"📹 VÍDEO {i}/{len(videos)}: {video['titulo']}")
        print(f"{'='*60}")

        # Obter transcrição
        print("📝 Obtendo transcrição...")
        transcricao = obter_transcricao(video['video_id'])

        if "Transcrição não disponível" in transcricao:
            print(f"⚠️ {transcricao}")
            resumo = "Transcrição não disponível para este vídeo."
        else:
            print(f"✅ Transcrição obtida ({len(transcricao)} caracteres)")

            # Gerar resumo
            print("🤖 Gerando resumo com Gemini...")
            resumo = resumir_com_gemini(transcricao, video['titulo'])
            print(f"✅ Resumo gerado")

        resumos.append({
            'numero': i,
            'titulo': video['titulo'],
            'url': video['url'],
            'resumo': resumo
        })

    return resumos

# Processar todos os vídeos
print("🚀 Iniciando processamento de todos os vídeos...\n")
resumos = processar_videos(videos)

🚀 Iniciando processamento de todos os vídeos...


📹 VÍDEO 1/5: COMO GANHAR DINHEIRO NA INTERNET? - CLIQUE PARA ASSISTIR #shorts
📝 Obtendo transcrição...
✅ Transcrição obtida (938 caracteres)
🤖 Gerando resumo com Gemini...
✅ Resumo gerado

📹 VÍDEO 2/5: COMO GANHAR DINHEIRO COM MARKETING DIGITAL? - CLIQUE PARA ASSISTIR #shorts
📝 Obtendo transcrição...
✅ Transcrição obtida (1734 caracteres)
🤖 Gerando resumo com Gemini...
✅ Resumo gerado

📹 VÍDEO 3/5: AINDA DÁ TEMPO DE COMEÇAR NO MARKETING DIGITAL?
📝 Obtendo transcrição...
✅ Transcrição obtida (6231 caracteres)
🤖 Gerando resumo com Gemini...
✅ Resumo gerado

📹 VÍDEO 4/5: o SEGREDO PRA MUDAR DE VIDA é NÃO TER MEDO DE ARRISCAR
📝 Obtendo transcrição...
✅ Transcrição obtida (8251 caracteres)
🤖 Gerando resumo com Gemini...
✅ Resumo gerado

📹 VÍDEO 5/5: VENDA PRODUTOS EM ESPANHOL E FATURE MILHÕES - CLIQUE PARA ASSISTIR #shorts
📝 Obtendo transcrição...
✅ Transcrição obtida (1383 caracteres)
🤖 Gerando resumo com Gemini...
✅ Resumo gerado


## 📄 Função 6: Unificar Resumos em Texto Único

In [15]:
def unificar_resumos(resumos):
    """
    Cria um texto unificado com todos os resumos
    """
    texto_unificado = "RESUMO DOS ÚLTIMOS 5 VÍDEOS DO CANAL\n\n"
    texto_unificado += "=" * 60 + "\n\n"

    for resumo in resumos:
        texto_unificado += f"VÍDEO {resumo['numero']}.\n"
        texto_unificado += f"TÍTULO: {resumo['titulo']}\n"
        texto_unificado += f"URL: {resumo['url']}\n\n"
        texto_unificado += f"RESUMO:\n{resumo['resumo']}\n\n"
        texto_unificado += "-" * 60 + "\n\n"

    return texto_unificado

# Criar texto unificado
texto_final = unificar_resumos(resumos)
print(texto_final)

# Salvar em arquivo
with open('resumos_unificados.txt', 'w', encoding='utf-8') as f:
    f.write(texto_final)

print("\n✅ Texto unificado salvo em 'resumos_unificados.txt'")

RESUMO DOS ÚLTIMOS 5 VÍDEOS DO CANAL


VÍDEO 1.
TÍTULO: COMO GANHAR DINHEIRO NA INTERNET? - CLIQUE PARA ASSISTIR #shorts
URL: https://www.youtube.com/watch?v=LuoaflbuHtg

RESUMO:
O vídeo encoraja a todos a explorar o potencial de ganho na internet, destacando que muitas pessoas comuns conseguiram sucesso. É feita uma analogia com jogadores de futebol da Série B, que, apesar de não serem "grandes gênios", podem ganhar salários expressivos (como R$70.000), sugerindo que no ambiente digital, indivíduos menos conhecidos também alcançam rendimentos significativos (R$40.000-R$45.000).

No entanto, o foco principal não deve ser nos números mais altos, mas sim em objetivos mais realistas e alcançáveis. A sugestão é tentar substituir um salário de um emprego indesejado, por exemplo, almejar ganhar R$2.000 na internet se esse for o valor atual. Essa abordagem visa tornar o objetivo mais tangível e motivador.

O narrador compartilha sua experiência pessoal, revelando que antes de começar a trabal

## 🔊 Função 7: Gerar Áudio com Gemini

In [21]:
def gerar_audio_gemini(texto):
    """
    Gera áudio a partir do texto usando Gemini Text-to-Speech
    Nota: Gemini não tem TTS nativo, usando alternativa
    """
    print("🔊 Gerando áudio...")


    try:
        from gtts import gTTS
        import os

        # Criar versão resumida para áudio (limite de caracteres)
        prompt = f"""
        Crie um roteiro de áudio narrado em português brasileiro, resumindo os principais pontos
        dos vídeos abaixo. O texto deve ser fluido e natural para ser lido em voz alta,
        com duração aproximada de 2-3 minutos.

        {texto}
        """

        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        texto_audio = resp.text


        # Gerar áudio
        tts = gTTS(text=texto_audio, lang='pt-br', slow=False)
        tts.save('audio.wav')

        print("✅ Áudio gerado com sucesso: audio.wav")
        return 'audio.wav', texto_audio

    except ImportError:
        print("📦 Instalando gTTS...")
        !pip install -q gtts
        from gtts import gTTS

        # Repetir processo
        model = genai.GenerativeModel('gemini-1.5-flash')
        prompt = f"""
        Crie um roteiro de áudio narrado em português brasileiro, resumindo os principais pontos
        dos vídeos abaixo. O texto deve ser fluido e natural para ser lido em voz alta,
        com duração aproximada de 2-3 minutos.

        {texto}
        """

        tts = gTTS(text=texto_audio, lang='pt-br', slow=False)
        tts.save('audio.wav')

        print("✅ Áudio gerado com sucesso: audio.wav")
        return 'audio.wav', texto_audio

# Gerar áudio
arquivo_audio, roteiro_audio = gerar_audio_gemini(texto_final)

# Reproduzir áudio no notebook
print("\n🎧 Reproduzindo áudio:")
display(Audio(arquivo_audio))

# Salvar roteiro
with open('roteiro_audio.txt', 'w', encoding='utf-8') as f:
    f.write(roteiro_audio)

print("\n✅ Roteiro do áudio salvo em 'roteiro_audio.txt'")

🔊 Gerando áudio...


✅ Áudio gerado com sucesso: audio.wav

🎧 Reproduzindo áudio:



✅ Roteiro do áudio salvo em 'roteiro_audio.txt'


## 📱 Função 8: Integração com Telegram

In [22]:
# Variável global para armazenar o chat_id
chat_id_destino = None

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handler para comando /start"""
    await update.message.reply_text(
        "🤖 Bot de Resumo de Vídeos YouTube\n\n"
        "Envie o link de um canal do YouTube (ex: https://www.youtube.com/@PrimoCast)\n"
        "e eu processarei os 5 últimos vídeos!"
    )

async def processar_mensagem(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handler para mensagens de texto"""
    global chat_id_destino
    chat_id_destino = update.effective_chat.id

    mensagem = update.message.text

    # Verificar se é um link do YouTube
    if 'youtube.com' in mensagem or 'youtu.be' in mensagem:
        await update.message.reply_text("🔄 Processando canal... Isso pode levar alguns minutos.")

        try:
            # Processar canal
            channel_id = extrair_channel_id(mensagem)
            videos = obter_ultimos_videos(channel_id)
            resumos = processar_videos(videos)
            texto_final = unificar_resumos(resumos)
            arquivo_audio, _ = gerar_audio_gemini(texto_final)

            # Enviar áudio
            await context.bot.send_audio(
                chat_id=chat_id_destino,
                audio=open(arquivo_audio, 'rb'),
                title="Resumo dos Vídeos",
                caption="🎧 Aqui está o resumo em áudio dos 5 últimos vídeos!"
            )

            # Enviar texto também
            # Dividir texto se for muito longo (limite Telegram: 4096 caracteres)
            if len(texto_final) > 4000:
                await context.bot.send_document(
                    chat_id=chat_id_destino,
                    document=open('resumos_unificados.txt', 'rb'),
                    caption="📄 Resumo completo em texto"
                )
            else:
                await update.message.reply_text(f"📄 Resumo:\n\n{texto_final}")

        except Exception as e:
            await update.message.reply_text(f"❌ Erro ao processar: {str(e)}")
    else:
        await update.message.reply_text(
            "⚠️ Por favor, envie um link válido do YouTube.\n"
            "Exemplo: https://www.youtube.com/@PrimoCast"
        )

def iniciar_bot_telegram():
    """Inicia o bot do Telegram"""
    print("🤖 Iniciando bot do Telegram...")

    # Criar aplicação
    app = Application.builder().token(TELEGRAM_BOT_TOKEN).build()

    # Adicionar handlers
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, processar_mensagem))

    print("✅ Bot configurado! Iniciando polling...")
    print("💡 Envie mensagens para o bot no Telegram para testar")
    print("⚠️ Pressione Ctrl+C para parar o bot")

    # Iniciar polling
    app.run_polling()

# Nota: Execute esta célula em uma aba separada ou use modo assíncrono
# Para testar manualmente, comente a linha abaixo e execute as funções diretamente
# iniciar_bot_telegram()

In [23]:
iniciar_bot_telegram()

🤖 Iniciando bot do Telegram...
✅ Bot configurado! Iniciando polling...
💡 Envie mensagens para o bot no Telegram para testar
⚠️ Pressione Ctrl+C para parar o bot


RuntimeError: Cannot close a running event loop

## 📤 Função 9: Enviar Áudio para Telegram (Manual)

In [25]:
async def enviar_audio_telegram(chat_id, arquivo_audio):
    """
    Envia o áudio gerado para um chat do Telegram
    """
    bot = Bot(token=TELEGRAM_BOT_TOKEN)

    print(f"📤 Enviando áudio para chat_id: {chat_id}")

    try:
        # Enviar áudio
        await bot.send_audio(
            chat_id=chat_id,
            audio=open(arquivo_audio, 'rb'),
            title="Resumo dos Últimos 5 Vídeos",
            caption="🎧 Aqui está o resumo em áudio dos vídeos do canal!"
        )

        # Enviar arquivo de texto também
        await bot.send_document(
            chat_id=chat_id,
            document=open('resumos_unificados.txt', 'rb'),
            caption="📄 Resumo completo em texto"
        )

        print("✅ Áudio e texto enviados com sucesso!")

    except Exception as e:
        print(f"❌ Erro ao enviar: {str(e)}")

# Para usar esta função, você precisa do chat_id
# Obtenha o chat_id enviando uma mensagem para o bot e usando:
# https://api.telegram.org/bot<TOKEN>/getUpdates

In [26]:
# Exemplo de uso:
SEU_CHAT_ID = 1529372943  # Substitua pelo seu chat_id
await enviar_audio_telegram(SEU_CHAT_ID, 'audio.wav')

📤 Enviando áudio para chat_id: 1529372943
✅ Áudio e texto enviados com sucesso!


## 🎯 Pipeline Completo - Execução Única

In [ ]:
def executar_pipeline_completo(url_canal, chat_id=None):
    """
    Executa o pipeline completo de processamento
    """
    print("\n" + "="*60)
    print("🚀 INICIANDO PIPELINE COMPLETO")
    print("="*60 + "\n")

    try:
        # 1. Extrair Channel ID
        print("📍 ETAPA 1: Extraindo Channel ID")
        channel_id = extrair_channel_id(url_canal)

        # 2. Obter últimos vídeos
        print("\n📍 ETAPA 2: Obtendo últimos 5 vídeos")
        videos = obter_ultimos_videos(channel_id, 5)

        # 3. Processar vídeos (transcrição + resumo)
        print("\n📍 ETAPA 3: Processando vídeos (transcrição + resumo)")
        resumos = processar_videos(videos)

        # 4. Unificar resumos
        print("\n📍 ETAPA 4: Unificando resumos")
        texto_final = unificar_resumos(resumos)

        # 5. Gerar áudio
        print("\n📍 ETAPA 5: Gerando áudio")
        arquivo_audio, roteiro = gerar_audio_gemini(texto_final)

        # 6. Enviar para Telegram (se chat_id fornecido)
        if chat_id:
            print("\n📍 ETAPA 6: Enviando para Telegram")
            import asyncio
            asyncio.run(enviar_audio_telegram(chat_id, arquivo_audio))

        print("\n" + "="*60)
        print("✅ PIPELINE CONCLUÍDO COM SUCESSO!")
        print("="*60)
        print(f"\n📁 Arquivos gerados:")
        print(f"  • audio.wav")
        print(f"  • resumos_unificados.txt")
        print(f"  • roteiro_audio.txt")

        return True

    except Exception as e:
        print(f"\n❌ ERRO NO PIPELINE: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# Executar pipeline
sucesso = executar_pipeline_completo(CANAL_YOUTUBE)

# Se quiser enviar para Telegram, descomente e adicione seu chat_id:
# sucesso = executar_pipeline_completo(CANAL_YOUTUBE, chat_id=SEU_CHAT_ID)

## 📊 Visualização dos Resultados

In [ ]:
# Exibir resumo final
print("\n📄 RESUMO FINAL:")
print("="*60)
print(texto_final)

# Exibir player de áudio
print("\n🎧 ÁUDIO GERADO:")
display(Audio('audio.wav'))

# Estatísticas
print("\n📊 ESTATÍSTICAS:")
print(f"  • Total de vídeos processados: {len(videos)}")
print(f"  • Tamanho do texto final: {len(texto_final)} caracteres")
print(f"  • Arquivo de áudio: audio.wav")

import os
if os.path.exists('audio.wav'):
    tamanho_audio = os.path.getsize('audio.wav') / 1024 / 1024
    print(f"  • Tamanho do áudio: {tamanho_audio:.2f} MB")

## 📥 Download dos Arquivos

In [ ]:
from google.colab import files

print("📥 Preparando arquivos para download...\n")

# Download do áudio
if os.path.exists('audio.wav'):
    print("⬇️ Baixando audio.wav...")
    files.download('audio.wav')

# Download do texto
if os.path.exists('resumos_unificados.txt'):
    print("⬇️ Baixando resumos_unificados.txt...")
    files.download('resumos_unificados.txt')

# Download do roteiro
if os.path.exists('roteiro_audio.txt'):
    print("⬇️ Baixando roteiro_audio.txt...")
    files.download('roteiro_audio.txt')

print("\n✅ Downloads iniciados!")

## 🔧 Utilitários e Testes

In [ ]:
# Obter chat_id do Telegram
def obter_chat_id():
    """
    Obtém o chat_id do Telegram
    Envie uma mensagem para o bot antes de executar
    """
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getUpdates"
    response = requests.get(url)
    data = response.json()

    if data['result']:
        for update in data['result']:
            if 'message' in update:
                chat_id = update['message']['chat']['id']
                username = update['message']['chat'].get('username', 'N/A')
                print(f"Chat ID: {chat_id}")
                print(f"Username: @{username}")
                return chat_id
    else:
        print("❌ Nenhuma mensagem encontrada. Envie uma mensagem para o bot primeiro!")
        return None

# Descomente para obter seu chat_id:
# meu_chat_id = obter_chat_id()

## 📝 Notas Finais

### Como Usar:

1. **Configure as credenciais** na seção de configuração
2. **Execute todas as células** em ordem
3. **Opção A - Automático via Bot:**
   - Execute a célula do bot do Telegram
   - Envie o link do canal para o bot
   - Receba o áudio automaticamente

4. **Opção B - Manual:**
   - Execute o pipeline completo
   - Faça download dos arquivos gerados
   - Envie manualmente para o Telegram

### Requisitos:

- **YouTube Data API Key**: https://console.cloud.google.com/
- **Gemini API Key**: https://aistudio.google.com/
- **Telegram Bot Token**: @BotFather no Telegram

### Limitações:

- Vídeos sem legendas/transcrições não podem ser processados
- APIs têm limites de uso (quotas)
- Áudio gerado pode ser grande para envio no Telegram (limite: 50MB)

### Troubleshooting:

- **Erro de API**: Verifique se as chaves estão corretas
- **Transcrição não disponível**: Vídeo não tem legendas
- **Timeout**: Canal com vídeos muito longos (ajuste quantidade)

